# LeetCode 4082 — Minimum Operations to Stabilize a Binary Tree (Hard)

Bài toán LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def stabilizeBinaryTree(self, root: Optional[TreeNode]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4082 (trên LeetCode bài này thuộc nhóm Premium).

Đây là một bài **quy hoạch động trên cây với trạng thái là giá trị số** — mà giá trị có thể nằm trong khoảng rộng `2·10^5`. Notebook dẫn bạn đi từng bước tới một kỹ thuật nâng cao nhưng rất đẹp: **slope trick** (thủ thuật độ dốc):

1. DP theo giá trị: đúng nhưng chậm.
2. Quan sát: mỗi hàm DP có **hình thung lũng (lồi)**, gấp khúc tại các số nguyên.
3. Lưu hàm lồi bằng **các điểm gãy** trong hai **heap**, thay vì lưu cả mảng giá trị.
4. Mọi phép toán của DP trở thành thao tác heap; gộp cây con theo kiểu **nhỏ vào lớn**.

Đừng lo nếu chưa từng nghe "slope trick" — ta sẽ xây nó từ những ví dụ số cụ thể.

---

## Sau notebook này, bạn sẽ nắm được

1. Dựng cây nhị phân từ danh sách level-order, duyệt cây không đệ quy.
2. Vì sao có thể giới hạn giá trị mới trong `[min val, max val]` (mẹo "kẹp giá trị").
3. DP theo giá trị `O(n · V)` và hình dạng lồi của các hàm DP.
4. Biểu diễn hàm lồi tuyến tính từng khúc bằng điểm gãy; ba phép toán: khởi tạo `|x - a|`, **mở rộng đáy**, **cộng hàm**.
5. Cài đặt slope trick với heap + offset lười + gộp nhỏ vào lớn: `O(n log² n)`.
6. Lỗi hay gặp, mẹo nhận diện (kể cả bài kinh điển "làm mảng không giảm"), bài tập.

# 1. Đề bài yêu cầu gì?

Cho gốc của một cây nhị phân, mỗi nút chứa một số nguyên.

Cây được gọi là **ổn định (stable)** nếu với **mọi** nút `u` và **mọi** con `c` của nó:

```text
|u.val - c.val| <= 1
```

Một **thao tác**: chọn một nút bất kỳ, tăng hoặc giảm giá trị của nó đi 1.

Trả về **số thao tác ít nhất** để cây ổn định.

(Tổng số thao tác = tổng `|giá trị mới - giá trị cũ|` trên mọi nút.)

---

## Ví dụ 1

```python
root = [5, 8, 3]   ->   3
```

```text
    5
   / \
  8   3
```

Giảm `8` xuống `6` (2 thao tác), tăng `3` lên `4` (1 thao tác). Tổng `3`.

## Ví dụ 2

```python
root = [2, -1, -4]   ->   5
```

Giảm gốc từ `2` xuống `-2` (4 thao tác), tăng `-4` lên `-3` (1 thao tác). Con trái `-1` đã thỏa `|-2 - (-1)| = 1`. Tổng `5`. Để ý: lời giải tối ưu **thay đổi cả gốc**, không chỉ các con.

## Ví dụ 3

```python
root = [0, -2, 2, -3, null, 1]   ->   3
```

```text
        0
      /   \
    -2     2
    /     /
  -3     1
```

Tăng `-2` lên `-1`, giảm `2` xuống `1`, tăng `-3` lên `-2`. Tổng `3`.

---

## Ràng buộc

```text
1 <= số nút <= 5 * 10^4
-10^5 <= Node.val <= 10^5
```

# 2. Kiến thức nền tảng

## 2.1. Cây nhị phân và dựng cây từ danh sách level-order

LeetCode cho cây dạng danh sách theo **từng tầng** (level-order), `None` là vị trí không có nút. Ta tự viết lớp `TreeNode` và hàm dựng cây để chạy thử.

In [1]:
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = [root]
    i = 1
    for node in queue:                       # queue dài dần trong lúc duyệt
        for side in ("left", "right"):
            if i >= len(values):
                return root
            if values[i] is not None:
                child = TreeNode(values[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root


def level_order(root: TreeNode) -> List[TreeNode]:
    order = [root]
    for u in order:
        for c in (u.left, u.right):
            if c:
                order.append(c)
    return order


root = build_tree([0, -2, 2, -3, None, 1])
print([u.val for u in level_order(root)])

[0, -2, 2, -3, 1]


`level_order` cho thứ tự **cha trước con**. Duyệt **ngược** danh sách này thì **con luôn được xử lý trước cha** — đúng thứ tự cần cho DP từ lá lên gốc, và không cần đệ quy (cây lệch có thể sâu `5·10^4` tầng).

## 2.2. Heap trong Python: `heapq`

- `heapq` là **min-heap**: `heap[0]` là phần tử nhỏ nhất; `heappush`, `heappop` tốn `O(log n)`.
- Muốn **max-heap**, lưu **số đối** `-x`.

In [2]:
import heapq

h = []
for x in [5, 1, 4]:
    heapq.heappush(h, x)
print("min-heap, nhỏ nhất:", h[0])

mh = []
for x in [5, 1, 4]:
    heapq.heappush(mh, -x)                    # lưu số đối
print("max-heap, lớn nhất:", -mh[0])

min-heap, nhỏ nhất: 1
max-heap, lớn nhất: 5


## 2.3. Mẹo "kẹp giá trị": chỉ cần xét giá trị trong `[lo, hi]`

Gọi `lo`, `hi` là giá trị nhỏ nhất và lớn nhất ban đầu trong cây. Với một cách gán bất kỳ, thay mỗi giá trị mới `x` bằng `clamp(x) = min(max(x, lo), hi)`:

- Ràng buộc vẫn thỏa: `|clamp(a) - clamp(b)| <= |a - b|` (kẹp không làm hai số xa nhau hơn).
- Chi phí không tăng: vì giá trị cũ `v` nằm trong `[lo, hi]`, `|clamp(x) - v| <= |x - v|`.

Vậy luôn tồn tại lời giải tối ưu với mọi giá trị mới trong `[lo, hi]`.

In [3]:
clamp = lambda x, lo, hi: min(max(x, lo), hi)
lo, hi = -4, 2
for a, b in [(-7, -6), (3, 4), (-5, 2)]:
    print(f"|{a} - {b}| = {abs(a - b)}  ->  sau khi kẹp: |{clamp(a, lo, hi)} - {clamp(b, lo, hi)}| = "
          f"{abs(clamp(a, lo, hi) - clamp(b, lo, hi))}")

|-7 - -6| = 1  ->  sau khi kẹp: |-4 - -4| = 0
|3 - 4| = 1  ->  sau khi kẹp: |2 - 2| = 0
|-5 - 2| = 7  ->  sau khi kẹp: |-4 - 2| = 6


# 3. Cách 1 — Vét cạn mọi cách gán giá trị

Thử mọi cách gán giá trị trong `[lo, hi]` cho từng nút — `(hi - lo + 1)^n` cách. Chỉ dùng cho cây rất nhỏ, làm chuẩn đối chiếu.

In [4]:
from itertools import product


def stabilize_bruteforce(root: TreeNode) -> int:
    nodes = level_order(root)
    lo = min(u.val for u in nodes)
    hi = max(u.val for u in nodes)
    pos = {id(u): k for k, u in enumerate(nodes)}
    edges = [(pos[id(u)], pos[id(c)]) for u in nodes for c in (u.left, u.right) if c]
    best = None
    for assign in product(range(lo, hi + 1), repeat=len(nodes)):
        if all(abs(assign[a] - assign[b]) <= 1 for a, b in edges):
            cost = sum(abs(assign[k] - nodes[k].val) for k in range(len(nodes)))
            best = cost if best is None else min(best, cost)
    return best


for vals in ([5, 8, 3], [2, -1, -4], [0, -2, 2, -3, None, 1]):
    print(vals, "->", stabilize_bruteforce(build_tree(vals)))

[5, 8, 3] -> 3
[2, -1, -4] -> 5
[0, -2, 2, -3, None, 1] -> 3


# 4. Cách 2 — DP theo giá trị: O(n · V)

Gọi:

```text
f_u(x) = số thao tác ít nhất cho cây con gốc u, BIẾT RẰNG giá trị mới của u là x
```

Nút `u` tốn `|x - u.val|`. Với mỗi con `c`, giá trị mới `y` của `c` phải thỏa `|y - x| <= 1`, tức `y ∈ {x-1, x, x+1}`, và ta chọn `y` tốt nhất:

```text
f_u(x) = |x - u.val| + Σ_{con c}  min( f_c(x-1), f_c(x), f_c(x+1) )
```

Đáp án: `min_x f_root(x)`.

## Tính tay ví dụ 1 (`[5, 8, 3]`, giá trị trong `[3, 8]`)

| x | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---:|---:|---:|---:|---:|---:|
| `f_8(x) = |x-8|` | 5 | 4 | 3 | 2 | 1 | 0 |
| `g_8(x) = min f_8 trong [x-1, x+1]` | 4 | 3 | 2 | 1 | 0 | 0 |
| `f_3(x) = |x-3|` | 0 | 1 | 2 | 3 | 4 | 5 |
| `g_3(x)` | 0 | 0 | 1 | 2 | 3 | 4 |
| `|x - 5|` | 2 | 1 | 0 | 1 | 2 | 3 |
| **`f_5(x)`** | 6 | 4 | **3** | 4 | 5 | 7 |

`min = 3` tại `x = 5` ✔.

In [5]:
def stabilize_dp_values(root: TreeNode, show: bool = False) -> int:
    nodes = level_order(root)
    lo = min(u.val for u in nodes)
    hi = max(u.val for u in nodes)
    V = hi - lo + 1
    f = {}
    for u in reversed(nodes):                       # con trước cha
        cur = [abs(lo + x - u.val) for x in range(V)]
        for c in (u.left, u.right):
            if c:
                fc = f.pop(id(c))
                for x in range(V):
                    cur[x] += min(fc[y] for y in (x - 1, x, x + 1) if 0 <= y < V)
        f[id(u)] = cur
        if show:
            print(f"nút {u.val:>3}: f = {cur}  (x từ {lo} tới {hi})")
    return min(f[id(root)])


print(stabilize_dp_values(build_tree([5, 8, 3]), show=True))

nút   3: f = [0, 1, 2, 3, 4, 5]  (x từ 3 tới 8)
nút   8: f = [5, 4, 3, 2, 1, 0]  (x từ 3 tới 8)
nút   5: f = [6, 4, 3, 4, 5, 7]  (x từ 3 tới 8)
3


Với `V = 2·10^5 + 1` và `n = 5·10^4`, DP theo giá trị cần `10^10` phép toán — quá chậm. Nhưng các mảng `f` in ở trên gợi ý một điều quan trọng...

---

# 5. Quan sát: mọi hàm `f_u` đều có hình **thung lũng (lồi)**

Nhìn `f_5 = [6, 4, 3, 4, 5, 7]`: giảm dần rồi tăng dần, và **độ dốc** (hiệu hai giá trị liên tiếp) **tăng dần**: `-2, -1, +1, +1, +2`. Hàm như vậy gọi là **lồi (convex)**.

Ba sự thật giữ cho tính lồi được bảo toàn qua DP:

1. `|x - a|` lồi (hình chữ V, độ dốc `-1` rồi `+1`).
2. **Tổng** các hàm lồi là hàm lồi (độ dốc cộng lại, vẫn tăng dần).
3. `g(x) = min(f(x-1), f(x), f(x+1))` với `f` lồi vẫn lồi: nó giống `f` nhưng **đáy được kéo rộng thêm 1 về mỗi phía** — phần dốc xuống dịch sang trái 1, phần dốc lên dịch sang phải 1.

Kiểm chứng sự thật 3 bằng số:

In [6]:
f8 = [5, 4, 3, 2, 1, 0, 1, 2, 3]                    # |x - 8| với x = 3..11
g8 = [min(f8[y] for y in (x - 1, x, x + 1) if 0 <= y < len(f8)) for x in range(len(f8))]
print("f =", f8, "-> đáy tại vị trí", f8.index(0))
print("g =", g8, "-> đáy rộng ra 3 vị trí")
slopes = lambda a: [a[i + 1] - a[i] for i in range(len(a) - 1)]
print("độ dốc của g:", slopes(g8))

f = [5, 4, 3, 2, 1, 0, 1, 2, 3] -> đáy tại vị trí 5
g = [4, 3, 2, 1, 0, 0, 0, 1, 2] -> đáy rộng ra 3 vị trí
độ dốc của g: [-1, -1, -1, -1, 0, 0, 1, 1]


Vì vậy **mọi `f_u` đều lồi** và **tuyến tính từng khúc**, gãy tại các số nguyên. Một hàm như thế được xác định hoàn toàn bởi: giá trị nhỏ nhất + **các điểm mà độ dốc tăng thêm 1** (gọi là **điểm gãy**). Số điểm gãy chỉ khoảng `2 ×` số nút trong cây con — nhỏ hơn nhiều so với `V = 2·10^5` giá trị.

---

# 6. Biểu diễn hàm lồi bằng điểm gãy (slope trick)

Ta viết mọi hàm lồi dưới dạng:

```text
f(x) = min_f  +  Σ_{l ∈ L} max(0, l - x)  +  Σ_{r ∈ R} max(0, x - r)
```

với điều kiện **mọi phần tử của `L` ≤ mọi phần tử của `R`**.

- Mỗi `l ∈ L` là một điểm gãy **bên trái đáy**: đi sang trái qua `l`, độ dốc giảm thêm 1.
- Mỗi `r ∈ R` là một điểm gãy **bên phải đáy**: đi sang phải qua `r`, độ dốc tăng thêm 1.
- Đáy (nơi đạt `min_f`) là đoạn `[max L, min R]`.

Ví dụ: `|x - 5|` có `L = {5}`, `R = {5}`, `min_f = 0`.

Kiểm tra với kết quả cuối cùng của ví dụ 1 (sẽ tính ở mục 8): `L = {2, 4, 5}`, `R = {5, 7, 9}`, `min_f = 3`.

In [7]:
def evaluate(min_f, L, R, x):
    return min_f + sum(max(0, l - x) for l in L) + sum(max(0, x - r) for r in R)


L, R, min_f = [2, 4, 5], [5, 7, 9], 3
print("từ điểm gãy :", [evaluate(min_f, L, R, x) for x in range(3, 9)])
print("DP theo giá trị: [6, 4, 3, 4, 5, 7]")

từ điểm gãy : [6, 4, 3, 4, 5, 7]
DP theo giá trị: [6, 4, 3, 4, 5, 7]


Khớp hoàn toàn. Giờ ta cần thực hiện ba phép toán của DP **trực tiếp trên điểm gãy**.

## 6.1. Khởi tạo `|x - a|`

`L = {a}`, `R = {a}`, `min_f = 0`.

## 6.2. Mở rộng đáy: `g(x) = min(f(x-1), f(x), f(x+1))`

Theo mục 5: phần bên trái dịch trái 1, phần bên phải dịch phải 1 → **mọi `l` giảm 1, mọi `r` tăng 1**, `min_f` giữ nguyên. Để không phải sửa từng phần tử, ta dùng **offset lười**: lưu `lo_off`, `ro_off` và hiểu giá trị thật là `l + lo_off`, `r + ro_off`. Mở rộng đáy chỉ là `lo_off -= 1`, `ro_off += 1` — `O(1)`!

## 6.3. Cộng thêm một hạng tử `max(0, x0 - x)` (một điểm gãy trái mới)

- Nếu `x0 <= min R`: chỉ cần thêm `x0` vào `L`. Điều kiện `L ≤ R` vẫn đúng, `min_f` không đổi (đáy cũ vẫn có `x >= x0`).
- Nếu `x0 > min R = y`: đáy dịch đi. Khi đó `min_f` tăng thêm `x0 - y`, ta thêm `x0` vào `R`, lấy `y` ra khỏi `R` và đưa `y` vào `L`. (Hiểu nôm na: `x0` "đẩy" điểm gãy `y` từ phải sang trái.)

Phép cộng `max(0, x - x0)` (điểm gãy phải mới) làm đối xứng.

## 6.4. Cộng hai hàm lồi

`f + g`: lấy `min_f + min_g`, rồi **cộng từng hạng tử** của `g` vào `f` bằng 6.3. Để nhanh, luôn chèn hàm **nhỏ hơn** (ít điểm gãy hơn) vào hàm **lớn hơn** — mỗi điểm gãy chỉ bị chuyển `O(log n)` lần (mỗi lần chuyển, kích thước tập chứa nó ít nhất gấp đôi).

In [8]:
class Convex:
    # f(x) = cost + Σ_{l∈L} max(0, l-x) + Σ_{r∈R} max(0, x-r),  max(L) <= min(R)
    def __init__(self, a: int):
        self.L = [-a]          # max-heap (lưu số đối), giá trị thật = -L[i] + lo_off
        self.R = [a]           # min-heap,               giá trị thật =  R[i] + ro_off
        self.lo_off = 0
        self.ro_off = 0
        self.cost = 0          # min_f

    def size(self):
        return len(self.L) + len(self.R)

    def widen(self):           # g(x) = min(f(x-1), f(x), f(x+1))
        self.lo_off -= 1
        self.ro_off += 1

    def add_left(self, x0: int):       # cộng max(0, x0 - x)
        if self.R and self.R[0] + self.ro_off < x0:
            y = heapq.heappop(self.R) + self.ro_off
            self.cost += x0 - y
            heapq.heappush(self.R, x0 - self.ro_off)
            x0 = y                          # y chuyển sang bên trái
        heapq.heappush(self.L, -(x0 - self.lo_off))

    def add_right(self, x0: int):      # cộng max(0, x - x0)
        if self.L and -self.L[0] + self.lo_off > x0:
            y = -heapq.heappop(self.L) + self.lo_off
            self.cost += y - x0
            heapq.heappush(self.L, -(x0 - self.lo_off))
            x0 = y                          # y chuyển sang bên phải
        heapq.heappush(self.R, x0 - self.ro_off)

    def absorb(self, other: "Convex"):   # self += other (other nhỏ hơn)
        self.cost += other.cost
        for v in other.L:
            self.add_left(-v + other.lo_off)
        for v in other.R:
            self.add_right(v + other.ro_off)

    def breakpoints(self):
        return (sorted(-v + self.lo_off for v in self.L), sorted(v + self.ro_off for v in self.R))


f = Convex(8)
f.widen()
print("widen(|x-8|): L, R =", f.breakpoints(), "cost =", f.cost)
h = Convex(5)
h.absorb(f)
print("|x-5| + widen(|x-8|): L, R =", h.breakpoints(), "cost =", h.cost)
print("giá trị x = 3..8:", [evaluate(h.cost, *h.breakpoints(), x) for x in range(3, 9)])

widen(|x-8|): L, R = ([7], [9]) cost = 0
|x-5| + widen(|x-8|): L, R = ([5, 5], [7, 9]) cost = 2
giá trị x = 3..8: [6, 4, 2, 2, 2, 3]


`|x - 5| + g_8(x)` với `x = 3..8`: `[2+4, 1+3, 0+2, 1+1, 2+0, 3+0] = [6, 4, 2, 2, 2, 3]` ✔ (đáy là đoạn `[5, 7]`, giá trị 2).

# 7. Cách 3 — Slope trick + gộp nhỏ vào lớn: O(n log² n)

Với mỗi nút `u` (con trước cha):

1. Bắt đầu từ `Convex(u.val)` (hàm `|x - u.val|`).
2. Với mỗi con `c`: lấy hàm của con, **mở rộng đáy**, rồi **cộng** vào hàm hiện tại (chèn hàm nhỏ vào hàm lớn — có thể phải **đổi vai** hai hàm).
3. Đáp án là `cost` (giá trị nhỏ nhất) của hàm ở gốc.

In [9]:
def stabilize_slope(root: TreeNode, verbose: bool = False) -> int:
    funcs = {}
    for u in reversed(level_order(root)):
        cur = Convex(u.val)
        for c in (u.left, u.right):
            if not c:
                continue
            child = funcs.pop(id(c))
            child.widen()
            if child.size() > cur.size():
                cur, child = child, cur          # luôn chèn hàm nhỏ vào hàm lớn
            cur.absorb(child)
        funcs[id(u)] = cur
        if verbose:
            L, R = cur.breakpoints()
            print(f"nút {u.val:>3}: L = {L}, R = {R}, min = {cur.cost}")
    return funcs[id(root)].cost


print(stabilize_slope(build_tree([5, 8, 3]), verbose=True))
print()
print(stabilize_slope(build_tree([2, -1, -4]), verbose=True))

nút   3: L = [3], R = [3], min = 0
nút   8: L = [8], R = [8], min = 0
nút   5: L = [2, 4, 5], R = [5, 7, 9], min = 3
3

nút  -4: L = [-4], R = [-4], min = 0
nút  -1: L = [-1], R = [-1], min = 0
nút   2: L = [-5, -3, -2], R = [0, 2, 2], min = 5
5


## Mô phỏng ví dụ 1

| Bước | Hàm | L | R | min |
|---|---|---|---|---:|
| lá 8 | `|x-8|` | `{8}` | `{8}` | 0 |
| lá 3 | `|x-3|` | `{3}` | `{3}` | 0 |
| mở rộng con 8 | `g_8` | `{7}` | `{9}` | 0 |
| mở rộng con 3 | `g_3` | `{2}` | `{4}` | 0 |
| gốc 5 = `|x-5| + g_8 + g_3` | `f_5` | `{2, 4, 5}` | `{5, 7, 9}` | **3** |

Đáp án `3` ✔. Đáy của `f_5` là đoạn `[max L, min R] = [5, 5]`: giá trị mới tối ưu của gốc là 5.

# 8. Phiên bản đúng format LeetCode

In [10]:
import heapq
from typing import Optional


class Solution:
    def stabilizeBinaryTree(self, root: Optional[TreeNode]) -> int:
        order = [root]
        for u in order:
            for c in (u.left, u.right):
                if c:
                    order.append(c)

        # trạng thái mỗi hàm: [L (max-heap số đối), lo_off, R (min-heap), ro_off, cost]
        def add_left(S, x):
            L, lo, R, ro, _ = S
            if R and R[0] + ro < x:
                y = heapq.heappop(R) + ro
                S[4] += x - y
                heapq.heappush(R, x - ro)
                x = y
            heapq.heappush(L, -(x - lo))

        def add_right(S, x):
            L, lo, R, ro, _ = S
            if L and -L[0] + lo > x:
                y = -heapq.heappop(L) + lo
                S[4] += y - x
                heapq.heappush(L, -(x - lo))
                x = y
            heapq.heappush(R, x - ro)

        state = {}
        for u in reversed(order):
            S = [[-u.val], 0, [u.val], 0, 0]
            for c in (u.left, u.right):
                if not c:
                    continue
                C = state.pop(id(c))
                C[1] -= 1                          # mở rộng đáy
                C[3] += 1
                if len(C[0]) + len(C[2]) > len(S[0]) + len(S[2]):
                    S, C = C, S
                S[4] += C[4]
                for v in C[0]:
                    add_left(S, -v + C[1])
                for v in C[2]:
                    add_right(S, v + C[3])
            state[id(u)] = S
        return state[id(root)][4]

In [11]:
sol = Solution()
tests = [
    ([5, 8, 3], 3, "ví dụ 1"),
    ([2, -1, -4], 5, "ví dụ 2"),
    ([0, -2, 2, -3, None, 1], 3, "ví dụ 3"),
    ([7], 0, "một nút"),
    ([3, 0], 2, "hai nút"),
    ([1, 2, 0, 3, 1, -1, 1], 0, "đã ổn định"),
]
for vals, expected, name in tests:
    result = sol.stabilizeBinaryTree(build_tree(vals))
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:14} -> {result}")

PASS | ví dụ 1        -> 3
PASS | ví dụ 2        -> 5
PASS | ví dụ 3        -> 3
PASS | một nút        -> 0
PASS | hai nút        -> 2
PASS | đã ổn định     -> 0


# 9. Vì sao đúng?

1. **DP đúng:** cây con của các con là độc lập khi đã biết giá trị mới `x` của `u`; ràng buộc chỉ giữa `u` và từng con → công thức ở mục 4.
2. **Biểu diễn đúng:** `|x - a|` có biểu diễn `L = R = {a}`. Mở rộng đáy của hàm lồi = dịch phần trái sang trái 1, phần phải sang phải 1 (mục 5), tức `L - 1`, `R + 1`. Cộng hàm = cộng từng hạng tử `max(0, ·)`; phép `add_left`/`add_right` giữ đúng giá trị hàm và điều kiện `max L <= min R` (mục 6.3).
3. **Đáp án:** giá trị nhỏ nhất của hàm ở gốc chính là `cost` vì mọi hạng tử `max(0, ·)` bằng 0 tại đáy `[max L, min R]`.

---

## Trực giác

> Mỗi hàm DP là một "thung lũng". Thay vì ghi độ cao tại từng điểm, ta chỉ ghi **những chỗ sườn núi gãy**. Kéo rộng đáy là xê dịch các chỗ gãy; ghép hai thung lũng là trộn các chỗ gãy lại.

# 10. Độ phức tạp

Gọi `n` = số nút, `V = max val - min val + 1`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Vét cạn | mọi cách gán | `O(V^n · n)` | `O(n)` |
| 2. DP theo giá trị | mảng `f_u` dài `V` | `O(n · V)` | `O(n · V)` (có thể giảm) |
| 3. Slope trick | heap điểm gãy + nhỏ vào lớn | `O(n log² n)` | `O(n)` |

Ở cách 3: mỗi điểm gãy bị chèn lại `O(log n)` lần (kích thước tập chứa nó gấp đôi mỗi lần), mỗi lần chèn `O(log n)` → `O(n log² n)`.

# 11. Đối chiếu ba cách trên cây ngẫu nhiên

In [12]:
import random


def random_tree_values(n, lo, hi):
    vals = [random.randint(lo, hi)]
    while sum(v is not None for v in vals) < n:
        vals.append(random.choice([None, random.randint(lo, hi), random.randint(lo, hi)]))
    return vals


random.seed(4082)
for _ in range(1000):
    vals = random_tree_values(random.randint(1, 6), -3, 3)
    r = build_tree(vals)
    expected = stabilize_bruteforce(r)
    assert stabilize_dp_values(r) == expected, vals
    assert stabilize_slope(r) == expected, vals
    assert sol.stabilizeBinaryTree(r) == expected, vals
for _ in range(300):
    vals = [random.randint(-40, 40) for _ in range(random.randint(1, 60))]
    r = build_tree(vals)
    assert stabilize_dp_values(r) == sol.stabilizeBinaryTree(r), vals
print("1000 cây nhỏ (so với vét cạn) + 300 cây vừa (so với DP theo giá trị): khớp!")

1000 cây nhỏ (so với vét cạn) + 300 cây vừa (so với DP theo giá trị): khớp!


In [13]:
import time

N = 5 * 10**4
big_balanced = build_tree([random.randint(-10**5, 10**5) for _ in range(N)])
chain_root = TreeNode(random.randint(-10**5, 10**5))
node = chain_root
for _ in range(N - 1):                           # cây lệch: sâu 5·10^4 tầng
    node.left = TreeNode(random.randint(-10**5, 10**5))
    node = node.left
for name, r in [("cây cân bằng", big_balanced), ("cây lệch", chain_root)]:
    t0 = time.perf_counter()
    res = sol.stabilizeBinaryTree(r)
    print(f"{name}: {res:,} thao tác ({time.perf_counter() - t0:.2f}s)")

cây cân bằng: 2,505,165,907 thao tác (0.20s)
cây lệch: 2,492,113,781 thao tác (0.06s)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Tham lam: giữ nguyên cha, chỉnh con cho gần cha

Ví dụ 2 cho thấy lời giải tối ưu phải **đổi cả gốc**. Chiến lược "duyệt từ gốc xuống, kéo mỗi con về sát cha" sai.

In [14]:
def wrong_greedy_top_down(root):
    total = 0
    new_val = {id(root): root.val}
    for u in level_order(root):
        for c in (u.left, u.right):
            if c:
                p = new_val[id(u)]
                target = min(max(c.val, p - 1), p + 1)      # kéo con vào [p-1, p+1]
                total += abs(target - c.val)
                new_val[id(c)] = target
    return total


r = build_tree([2, -1, -4])
print("Sai :", wrong_greedy_top_down(r))
print("Đúng:", sol.stabilizeBinaryTree(r))

Sai : 7
Đúng: 5


## Lỗi 2 — Quên mở rộng đáy (bắt con bằng đúng giá trị cha)

Nếu bỏ bước `widen`, ta đang giải bài "mọi nút phải **bằng nhau**" — tốn hơn nhiều.

In [15]:
def wrong_no_widen(root):
    funcs = {}
    for u in reversed(level_order(root)):
        cur = Convex(u.val)
        for c in (u.left, u.right):
            if c:
                child = funcs.pop(id(c))               # SAI: thiếu child.widen()
                if child.size() > cur.size():
                    cur, child = child, cur
                cur.absorb(child)
        funcs[id(u)] = cur
    return funcs[id(root)].cost


r = build_tree([5, 8, 3])
print("Sai :", wrong_no_widen(r))
print("Đúng:", sol.stabilizeBinaryTree(r))

Sai : 5
Đúng: 3


## Lỗi 3 — `add_left` không "đẩy" điểm gãy khi `x0 > min R`

Nếu luôn chỉ đẩy `x0` vào `L`, điều kiện `max L <= min R` bị phá và `cost` không được cập nhật → kết quả sai.

In [16]:
class WrongConvex(Convex):
    def add_left(self, x0):
        heapq.heappush(self.L, -(x0 - self.lo_off))          # SAI: bỏ qua trường hợp x0 > min R

    def add_right(self, x0):
        heapq.heappush(self.R, x0 - self.ro_off)


def wrong_no_swap(root):
    funcs = {}
    for u in reversed(level_order(root)):
        cur = WrongConvex(u.val)
        for c in (u.left, u.right):
            if c:
                child = funcs.pop(id(c))
                child.widen()
                if child.size() > cur.size():
                    cur, child = child, cur
                cur.absorb(child)
        funcs[id(u)] = cur
    return funcs[id(root)].cost


r = build_tree([5, 8, 3])
print("Sai :", wrong_no_swap(r))
print("Đúng:", sol.stabilizeBinaryTree(r))

Sai : 0
Đúng: 3


## Lỗi 4 — Luôn chèn hàm của con vào hàm của cha (không đổi vai)

Vẫn đúng, nhưng với cây lệch (mỗi nút một con), hàm của con lớn dần mà vẫn bị chèn lại toàn bộ ở mỗi tầng → `O(n²)`. Đo thử:

In [17]:
def stabilize_no_small_to_large(root):
    funcs = {}
    for u in reversed(level_order(root)):
        cur = Convex(u.val)
        for c in (u.left, u.right):
            if c:
                child = funcs.pop(id(c))
                child.widen()
                cur.absorb(child)                          # luôn chèn con vào cha
        funcs[id(u)] = cur
    return funcs[id(root)].cost


def chain(n):
    r = TreeNode(random.randint(-100, 100))
    t = r
    for _ in range(n - 1):
        t.left = TreeNode(random.randint(-100, 100))
        t = t.left
    return r


for n_ in [300, 600, 1200]:
    r = chain(n_)
    t0 = time.perf_counter(); a = stabilize_no_small_to_large(r); t1 = time.perf_counter() - t0
    t0 = time.perf_counter(); b = sol.stabilizeBinaryTree(r); t2 = time.perf_counter() - t0
    print(f"chuỗi {n_}: không đổi vai {t1:.2f}s | nhỏ vào lớn {t2:.3f}s | kết quả {a == b}")

chuỗi 300: không đổi vai 0.01s | nhỏ vào lớn 0.000s | kết quả True
chuỗi 600: không đổi vai 0.04s | nhỏ vào lớn 0.001s | kết quả True


chuỗi 1200: không đổi vai 0.15s | nhỏ vào lớn 0.001s | kết quả True


## Lỗi 5 — Dùng đệ quy cho cây lệch

Cây sâu `5·10^4` tầng làm DFS đệ quy báo `RecursionError`. Dùng thứ tự level-order đảo ngược như trên.

## Lỗi 6 — Quên offset khi lấy giá trị thật ra khỏi heap

Mọi phần tử trong heap đều phải cộng offset (`-v + lo_off` với `L`, `v + ro_off` với `R`) trước khi so sánh hoặc chuyển sang hàm khác. Quên offset là nguồn lỗi phổ biến nhất khi cài slope trick.

# 13. Mẹo nhận diện

1. Chi phí dạng **`Σ |giá trị mới - giá trị cũ|`** + ràng buộc **giữa các phần tử kề nhau** (`|a - b| <= k`, `a <= b`, ...) → DP theo giá trị, các hàm DP **lồi** → nghĩ tới **slope trick**.
2. Các phép toán thường gặp trên hàm lồi và cách làm bằng điểm gãy:

| Phép toán trên hàm | Trên điểm gãy |
|---|---|
| cộng `|x - a|` | `add_left(a)` + `add_right(a)` |
| `min` trong cửa sổ `[x-k, x+k]` | `L -= k`, `R += k` (offset lười) |
| `min` tiền tố (`g(x) = min_{y<=x} f(y)`) | xóa toàn bộ `R` |
| cộng hai hàm | chèn nhỏ vào lớn |

3. Trên cây → gộp các hàm của con theo kiểu **nhỏ vào lớn**.

Bài kinh điển cùng kỹ thuật: *Make Array Non-decreasing or Non-increasing* (LeetCode 2263), bài "Sequence" (làm mảng không giảm với tổng `|thay đổi|` nhỏ nhất).

# 14. Biến thể

## Biến thể 1 — Mảng: `|a[i] - a[i+1]| <= k`

Mảng là một cây lệch. Ràng buộc `<= k` → mở rộng đáy `k` đơn vị: `L -= k`, `R += k`.

In [18]:
def stabilize_array(a: List[int], k: int) -> int:
    f = Convex(a[0])
    for x in a[1:]:
        f.lo_off -= k                         # mở rộng đáy k đơn vị
        f.ro_off += k
        f.add_left(x)                         # cộng |y - x|
        f.add_right(x)
    return f.cost


def brute_array(a, k):
    lo, hi = min(a), max(a)
    best = None
    for b in product(range(lo, hi + 1), repeat=len(a)):
        if all(abs(b[i] - b[i + 1]) <= k for i in range(len(a) - 1)):
            c = sum(abs(b[i] - a[i]) for i in range(len(a)))
            best = c if best is None else min(best, c)
    return best


random.seed(1)
for _ in range(300):
    arr = [random.randint(-4, 4) for _ in range(random.randint(1, 5))]
    kk = random.randint(0, 2)
    assert stabilize_array(arr, kk) == brute_array(arr, kk)
print(stabilize_array([1, 10, 2], 2), "| khớp brute force trên 300 mảng")

7 | khớp brute force trên 300 mảng


## Biến thể 2 — Làm mảng **không giảm** với ít thao tác nhất (bài kinh điển)

Ràng buộc `a[i] <= a[i+1]` → hàm DP chuyển tiếp là `g(x) = min_{y <= x} f(y)` (xóa phần dốc lên, tức bỏ `R`). Khi chỉ cần giữ `L`, thuật toán còn đúng **5 dòng** với một max-heap:

In [19]:
def min_ops_non_decreasing(a: List[int]) -> int:
    heap, cost = [], 0
    for x in a:
        heapq.heappush(heap, -x)
        if -heap[0] > x:                      # có phần tử trước lớn hơn x
            cost += -heap[0] - x
            heapq.heapreplace(heap, -x)       # "hạ" phần tử lớn nhất xuống x
    return cost


def brute_non_dec(a):
    lo, hi = min(a), max(a)
    return min(sum(abs(b[i] - a[i]) for i in range(len(a)))
               for b in product(range(lo, hi + 1), repeat=len(a))
               if all(b[i] <= b[i + 1] for i in range(len(a) - 1)))


random.seed(2)
for _ in range(300):
    arr = [random.randint(0, 5) for _ in range(random.randint(1, 5))]
    assert min_ops_non_decreasing(arr) == brute_non_dec(arr)
print(min_ops_non_decreasing([3, 2, 4, 5, 1]), "| khớp brute force trên 300 mảng")

5 | khớp brute force trên 300 mảng


# 15. Template tư duy

```python
# Slope trick: f(x) = cost + Σ_{l∈L} (l-x)^+ + Σ_{r∈R} (x-r)^+
# L: max-heap (số đối) + lo_off ; R: min-heap + ro_off ; luôn max L <= min R
init |x-a|      : L = {a}, R = {a}
window min ±k   : lo_off -= k ; ro_off += k
add (x0 - x)^+  : nếu x0 > min R: cost += x0 - minR, R: thay minR bằng x0, x0 = minR cũ ; đẩy x0 vào L
add (x - x0)^+  : đối xứng
f + g           : cost += g.cost ; chèn từng điểm gãy của hàm nhỏ vào hàm lớn
answer          : cost
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Cây `[3, 0]` (gốc 3, con trái 0). Lập bảng DP theo giá trị với `x ∈ [0, 3]`: `f_0`, `g_0`, `|x - 3|`, `f_3`. Đáp án là bao nhiêu? Sau đó đối chiếu với điểm gãy ở cell dưới.

In [20]:
# Kiểm tra Bài 1
print(stabilize_dp_values(build_tree([3, 0]), show=True))
print(stabilize_slope(build_tree([3, 0]), verbose=True))

nút   0: f = [0, 1, 2, 3]  (x từ 0 tới 3)
nút   3: f = [3, 2, 2, 2]  (x từ 0 tới 3)
2
nút   0: L = [0], R = [0], min = 0
nút   3: L = [-1, 1], R = [3, 3], min = 2
2


## Bài 2 — Dự đoán output

```python
f = Convex(4)
f.widen(); f.widen()
print(f.breakpoints())
f.add_left(7)
print(f.breakpoints(), f.cost)
```

In [21]:
f = Convex(4)
f.widen(); f.widen()
print(f.breakpoints())
f.add_left(7)
print(f.breakpoints(), f.cost)

([2], [6])
([2, 6], [7]) 1


## Bài 3 — Tự viết `add_left` và `add_right`

Hoàn thiện hai hàm trên một trạng thái dạng `[L, lo_off, R, ro_off, cost]`.

In [22]:
def exercise_3_add_left(S, x0):
    # TODO: nếu R khác rỗng và min R (thật) < x0: cost += x0 - y, thay y bằng x0 trong R, x0 = y
    #       rồi đẩy x0 vào L (nhớ offset và số đối)
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


def exercise_3_add_right(S, x0):
    # TODO: đối xứng
    raise NotImplementedError

## Bài 4 — Biến thể: ràng buộc `|u.val - c.val| <= k`

Sửa lời giải để nhận thêm tham số `k` (mở rộng đáy `k` đơn vị). Kiểm tra với vét cạn.

```python
stabilize_k(build_tree([5, 8, 3]), 1)   -> 3
stabilize_k(build_tree([5, 8, 3]), 3)   -> 0
```

In [23]:
def exercise_4_stabilize_k(root: TreeNode, k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_stabilize_k(build_tree([5, 8, 3]), 1) == 3
# assert exercise_4_stabilize_k(build_tree([5, 8, 3]), 3) == 0

## Bài 5 — Làm mảng không tăng

Viết hàm tính số thao tác ít nhất để mảng **không tăng** (`a[i] >= a[i+1]`). Gợi ý: đảo dấu hoặc đảo mảng rồi dùng `min_ops_non_decreasing`.

```python
min_ops_non_increasing([1, 3, 2])   -> 2    # ví dụ [2, 2, 2]
```

In [24]:
def exercise_5_min_ops_non_increasing(a: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_min_ops_non_increasing([1, 3, 2]) == 2

## Bài 6 — Mini challenge: chỉ ra giá trị mới của từng nút

Với cây nhỏ (giá trị trong khoảng hẹp), dùng DP theo giá trị để **truy vết** và trả về danh sách giá trị mới theo thứ tự level-order. Kiểm tra: cây mới ổn định và tổng thay đổi bằng đáp án.

Gợi ý: chọn `x*` tối ưu ở gốc; với mỗi con, chọn `y ∈ {x*-1, x*, x*+1}` có `f_c(y)` nhỏ nhất; đi xuống tiếp.

In [25]:
def exercise_6_new_values(root: TreeNode) -> List[int]:
    # TODO
    raise NotImplementedError


# r = build_tree([0, -2, 2, -3, None, 1])
# new = exercise_6_new_values(r)
# assert sum(abs(a - u.val) for a, u in zip(new, level_order(r))) == 3

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: giá trị thật của `R[0]` là `R[0] + ro_off`; của `L[0]` là `-L[0] + lo_off`.
- Mức 2: khi lưu một giá trị thật `v` vào `R`: `heappush(R, v - ro_off)`; vào `L`: `heappush(L, -(v - lo_off))`.
- Mức 3: chép cấu trúc của `Convex.add_left` nhưng thao tác trên `S[0..4]`.

**Bài 4**

- Mức 1: thay `widen()` (±1) bằng `lo_off -= k; ro_off += k`.
- Mức 2: phần còn lại giữ nguyên.
- Mức 3: vét cạn để kiểm tra: thay điều kiện `<= 1` bằng `<= k`.

**Bài 5**

- Mức 1: mảng không tăng ⇔ mảng đảo ngược không giảm.
- Mức 2: `min_ops_non_decreasing(a[::-1])`.
- Mức 3: hoặc `min_ops_non_decreasing([-x for x in a])`.

**Bài 6**

- Mức 1: lưu lại mảng `f_u` của mọi nút (đừng `pop`).
- Mức 2: `x*` = `argmin f_root`.
- Mức 3: duyệt level-order, mỗi con chọn `y` tốt nhất trong `{x-1, x, x+1}` (giới hạn trong `[lo, hi]`).

# 17. Đáp án tham khảo

## Đáp án Bài 1

| x | 0 | 1 | 2 | 3 |
|---|---:|---:|---:|---:|
| `f_0 = |x|` | 0 | 1 | 2 | 3 |
| `g_0` | 0 | 0 | 1 | 2 |
| `|x - 3|` | 3 | 2 | 1 | 0 |
| `f_3` | 3 | 2 | 2 | 2 |

Đáp án `2` (ví dụ đổi gốc thành 1, hoặc con thành 2). Điểm gãy cuối cùng: `L = {-1, 1}`, `R = {3, 3}`, `min = 2` — đáy là đoạn `[1, 3]`.

## Đáp án Bài 2

```text
([2], [6])
([2, 6], [7]) 1
```

Sau hai lần mở rộng, `|x - 4|` thành hàm có đáy `[2, 6]`. Cộng `max(0, 7 - x)`: vì `7 > min R = 6`, chi phí tăng `1`, điểm gãy `6` chuyển sang trái, `7` vào bên phải.

In [26]:
# Đáp án Bài 3
def answer_3_add_left(S, x0):
    L, lo, R, ro, _ = S
    if R and R[0] + ro < x0:
        y = heapq.heappop(R) + ro
        S[4] += x0 - y
        heapq.heappush(R, x0 - ro)
        x0 = y
    heapq.heappush(L, -(x0 - lo))


def answer_3_add_right(S, x0):
    L, lo, R, ro, _ = S
    if L and -L[0] + lo > x0:
        y = -heapq.heappop(L) + lo
        S[4] += y - x0
        heapq.heappush(L, -(x0 - lo))
        x0 = y
    heapq.heappush(R, x0 - ro)


S = [[-4], -2, [4], 2, 0]                 # |x-4| đã mở rộng 2 lần
answer_3_add_left(S, 7)
print("cost =", S[4], "| L =", sorted(-v + S[1] for v in S[0]), "| R =", sorted(v + S[3] for v in S[2]))

cost = 1 | L = [2, 6] | R = [7]


In [27]:
# Đáp án Bài 4
def answer_4_stabilize_k(root: TreeNode, k: int) -> int:
    funcs = {}
    for u in reversed(level_order(root)):
        cur = Convex(u.val)
        for c in (u.left, u.right):
            if c:
                child = funcs.pop(id(c))
                child.lo_off -= k
                child.ro_off += k
                if child.size() > cur.size():
                    cur, child = child, cur
                cur.absorb(child)
        funcs[id(u)] = cur
    return funcs[id(root)].cost


def brute_k(root, k):
    nodes = level_order(root)
    lo, hi = min(u.val for u in nodes), max(u.val for u in nodes)
    pos = {id(u): i for i, u in enumerate(nodes)}
    edges = [(pos[id(u)], pos[id(c)]) for u in nodes for c in (u.left, u.right) if c]
    return min(sum(abs(b[i] - nodes[i].val) for i in range(len(nodes)))
               for b in product(range(lo, hi + 1), repeat=len(nodes))
               if all(abs(b[x] - b[y]) <= k for x, y in edges))


assert answer_4_stabilize_k(build_tree([5, 8, 3]), 1) == 3
assert answer_4_stabilize_k(build_tree([5, 8, 3]), 3) == 0
random.seed(4)
for _ in range(300):
    r = build_tree(random_tree_values(random.randint(1, 5), -3, 3))
    kk = random.randint(0, 2)
    assert answer_4_stabilize_k(r, kk) == brute_k(r, kk)
print("Bài 4: PASS (khớp vét cạn trên 300 cây)")

Bài 4: PASS (khớp vét cạn trên 300 cây)


In [28]:
# Đáp án Bài 5
def answer_5_min_ops_non_increasing(a: List[int]) -> int:
    return min_ops_non_decreasing(a[::-1])


assert answer_5_min_ops_non_increasing([1, 3, 2]) == 2
print("Bài 5: PASS")

Bài 5: PASS


In [29]:
# Đáp án Bài 6
def answer_6_new_values(root: TreeNode) -> List[int]:
    nodes = level_order(root)
    lo, hi = min(u.val for u in nodes), max(u.val for u in nodes)
    V = hi - lo + 1
    f = {}
    for u in reversed(nodes):
        cur = [abs(lo + x - u.val) for x in range(V)]
        for c in (u.left, u.right):
            if c:
                for x in range(V):
                    cur[x] += min(f[id(c)][y] for y in (x - 1, x, x + 1) if 0 <= y < V)
        f[id(u)] = cur
    choice = {id(root): min(range(V), key=lambda x: f[id(root)][x])}
    for u in nodes:
        x = choice[id(u)]
        for c in (u.left, u.right):
            if c:
                choice[id(c)] = min((y for y in (x - 1, x, x + 1) if 0 <= y < V), key=lambda y: f[id(c)][y])
    return [lo + choice[id(u)] for u in nodes]


r = build_tree([0, -2, 2, -3, None, 1])
new = answer_6_new_values(r)
print("giá trị mới:", new)
nodes = level_order(r)
pos = {id(u): i for i, u in enumerate(nodes)}
assert all(abs(new[pos[id(u)]] - new[pos[id(c)]]) <= 1 for u in nodes for c in (u.left, u.right) if c)
assert sum(abs(a - u.val) for a, u in zip(new, nodes)) == 3
print("Bài 6: PASS")

giá trị mới: [-1, -2, 0, -3, 1]
Bài 6: PASS


# 18. Tổng kết

**Minimum Operations to Stabilize a Binary Tree** là cửa ngõ vào **slope trick**:

1. DP theo giá trị: `f_u(x) = |x - val| + Σ min(f_c(x-1), f_c(x), f_c(x+1))`.
2. Mọi `f_u` lồi, tuyến tính từng khúc → lưu bằng **điểm gãy** trong hai heap `L`, `R`.
3. Mở rộng đáy = offset lười; cộng hàm = chèn điểm gãy (đẩy qua lại giữa `L` và `R` khi cần); trên cây gộp **nhỏ vào lớn**.
4. Đáp án = giá trị đáy `cost` ở gốc.

## Độ phức tạp

```text
Time:  O(n log² n)
Space: O(n)
```

# 19. Checklist trước khi submit

- [ ] Duyệt cây **không đệ quy** (level-order đảo ngược)?
- [ ] Mỗi nút khởi tạo `L = R = {val}`, `cost = 0`?
- [ ] Hàm của con được **mở rộng đáy** (`lo_off -= 1`, `ro_off += 1`) trước khi gộp?
- [ ] `add_left` / `add_right` xử lý trường hợp vượt qua đáy (đẩy điểm gãy, cộng `cost`)?
- [ ] Luôn cộng/trừ **offset** khi đọc/ghi heap?
- [ ] Gộp **nhỏ vào lớn** (đổi vai khi hàm con lớn hơn)?
- [ ] Cộng `cost` của hàm nhỏ vào hàm lớn khi gộp?